# UK Geographic Customer Opportunity Segmentation

## 1. Setup

### 1.1 Imports and project paths

In [1]:
from pathlib import Path
from urllib.parse import urlencode

import pandas as pd
import requests

# Display enough rows/columns to inspect the downloaded inputs without changing the data.
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)

# Toggle this when rerunning the notebook.
# False = use the CSVs already saved in data/input/ and do not call the Nomis API.
# True = query Nomis again and refresh the CSVs in data/input/.
QUERY_NOMIS_DATA = False

# Centroids are useful for later maps and attraction-distance calculations.
# False = use the cached centroid CSV if it exists.
# True = refresh the centroid CSV from the ONS Open Geography Portal.
REFRESH_MSOA_CENTROIDS = False

# The notebook lives in notebooks/, so the repository root is one level up.
# If you run this from a different working directory, adjust PROJECT_ROOT once here.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Store raw public inputs separately from later processed outputs.
DATA_INPUT_DIR = PROJECT_ROOT / "data" / "input"
DATA_INPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_INPUT_DIR

PosixPath('/Users/ChanB01/GitHub/uk-demographic-clustering/data/input')

## 2. Query Public MSOA Data

MSOA is the preferred geography because it is granular enough to show local opportunity, but still large enough to be explainable and reasonably stable.

For this prototype we start with Census 2021 tables from Nomis, the ONS-supported public API for England and Wales Census data. The raw files are saved as CSVs in `data/input/` so later notebook sections can clean, join, and engineer modelling features from a stable local copy.

#### Source links and sourcing judgement

- ONS Census 2021 overview: https://www.ons.gov.uk/census
- Nomis Census 2021 topic summaries source page: https://www.nomisweb.co.uk/sources/census_2021_ts
- Nomis Census 2021 approximated social grade source page: https://www.nomisweb.co.uk/sources/census_2021_asg
- Nomis API dataset catalogue: https://www.nomisweb.co.uk/api/v01/dataset/def.sdmx.json
- Nomis geography selector metadata for Census 2021 MSOAs: https://www.nomisweb.co.uk/api/v01/dataset/NM_2020_1/geography/TYPE.def.sdmx.json

The key judgement is that these sources are public, official, reproducible, and available at MSOA level for England and Wales. Census 2021 is not perfect for current-year population sizing, but it is a strong baseline for detailed demographic structure, household composition, car access, population density, deprivation dimensions, and social grade proxies. Later sections can blend in newer ONS population estimates if we want a more current market-size denominator.

The Nomis query method is transparent: each request uses a dataset ID, `geography=TYPE152` for 2021 MSOAs, a table-specific category dimension, `measures=20100` for raw counts/values, plus `recordlimit` and `recordoffset` pagination so the full table is downloaded rather than only the first page.

Rerun control: use the `QUERY_NOMIS_DATA` toggle in Section 1. Set it to `True` only when you want to refresh the raw CSVs from Nomis. Leave it as `False` for normal reruns so the notebook uses the cached files already saved in `data/input/`.

### 2.1 Public data configuration

Nomis uses dataset IDs for Census tables and dimension names for each table category. We query 2021 MSOAs using `TYPE152`, which is the Nomis geography selector for 2021 middle-layer super output areas in England and Wales.

These raw tables give us the main demand-side ingredients requested in the brief: population, age structure, family/household composition, car access, deprivation proxy, density, and an affluence proxy.

#### Table metadata links

Each link below is the Nomis metadata endpoint for the table used in this notebook. The metadata confirms the dataset ID, table description, geography availability, category dimension name, and measures.

- `NM_2020_1` - TS007A Age by five-year age bands: https://www.nomisweb.co.uk/api/v01/dataset/NM_2020_1/def.sdmx.json
- `NM_2023_1` - TS003 Household composition: https://www.nomisweb.co.uk/api/v01/dataset/NM_2023_1/def.sdmx.json
- `NM_2031_1` - TS011 Households by deprivation dimensions: https://www.nomisweb.co.uk/api/v01/dataset/NM_2031_1/def.sdmx.json
- `NM_2063_1` - TS045 Car or van availability: https://www.nomisweb.co.uk/api/v01/dataset/NM_2063_1/def.sdmx.json
- `NM_2026_1` - TS006 Population density: https://www.nomisweb.co.uk/api/v01/dataset/NM_2026_1/def.sdmx.json
- `NM_2312_1` - SG002 Approximated Social Grade: https://www.nomisweb.co.uk/api/v01/dataset/NM_2312_1/def.sdmx.json

Representative CSV query pattern:

`https://www.nomisweb.co.uk/api/v01/dataset/{dataset_id}.data.csv?date=latest&geography=TYPE152&{category_dimension}={category_selector}&measures=20100&recordlimit=25000&recordoffset=0`

Example for the first page of the age table:

`https://www.nomisweb.co.uk/api/v01/dataset/NM_2020_1.data.csv?date=latest&geography=TYPE152&c2021_age_19=0...18&measures=20100&recordlimit=25000&recordoffset=0`

In [2]:
# Base URL for the Nomis API.
# Documentation/metadata is exposed from the same public endpoint as the CSV data.
NOMIS_BASE_URL = "https://www.nomisweb.co.uk/api/v01/dataset"

# Nomis geography selector for all 2021 MSOAs in England and Wales.
# The project brief says to use MSOA if practical; Census 2021 MSOA data is practical here.
MSOA_2021_GEOGRAPHY = "TYPE152"

# The Nomis measure code 20100 means raw counts/values.
# We avoid percentage downloads here so all percentage features can be derived consistently later.
VALUE_MEASURE = "20100"

# Nomis returns large requests in pages. Keep the page size explicit so we do not
# accidentally save only the first page of a full-MSOA table.
NOMIS_PAGE_SIZE = 25_000

# Each table is downloaded in long format exactly as Nomis provides it.
# Later sections can pivot/select fields without needing to call the API again.
NOMIS_TABLES = [
    {
        "dataset_id": "NM_2020_1",
        "table_name": "TS007A - Age by five-year age bands",
        "category_dimension": "c2021_age_19",
        "category_selector": "0...18",
        "output_file": "msoa_age_5yr_census_2021.csv",
        "why_needed": "Population, children, young adults, and working-age parent proxy.",
    },
    {
        "dataset_id": "NM_2023_1",
        "table_name": "TS003 - Household composition",
        "category_dimension": "c2021_hhcomp_15",
        "category_selector": "0...14",
        "output_file": "msoa_household_composition_census_2021.csv",
        "why_needed": "Family household potential and household base counts.",
    },
    {
        "dataset_id": "NM_2031_1",
        "table_name": "TS011 - Households by deprivation dimensions",
        "category_dimension": "c2021_dep_6",
        "category_selector": "0...5",
        "output_file": "msoa_household_deprivation_census_2021.csv",
        "why_needed": "Public deprivation proxy for spending-power assumptions.",
    },
    {
        "dataset_id": "NM_2063_1",
        "table_name": "TS045 - Car or van availability",
        "category_dimension": "c2021_cars_5",
        "category_selector": "0...4",
        "output_file": "msoa_car_van_availability_census_2021.csv",
        "why_needed": "Car access proxy for leisure-trip accessibility.",
    },
    {
        "dataset_id": "NM_2026_1",
        "table_name": "TS006 - Population density",
        "category_dimension": "cell",
        "category_selector": "0",
        "output_file": "msoa_population_density_census_2021.csv",
        "why_needed": "Urban/density context for segmentation.",
    },
    {
        "dataset_id": "NM_2312_1",
        "table_name": "SG002 - Approximated Social Grade",
        "category_dimension": "c2021_asg_5",
        "category_selector": "0...4",
        "output_file": "msoa_approximated_social_grade_census_2021.csv",
        "why_needed": "Affluence/spending-power proxy for the prototype.",
    },
]

pd.DataFrame(NOMIS_TABLES)[["dataset_id", "table_name", "output_file", "why_needed"]]

,dataset_id,table_name,output_file,why_needed
0,NM_2020_1,TS007A - Age by five-year age bands,msoa_age_5yr_census_2021.csv,"Population, children, young adults, and working-age parent proxy."
1,NM_2023_1,TS003 - Household composition,msoa_household_composition_census_2021.csv,Family household potential and household base counts.
2,NM_2031_1,TS011 - Households by deprivation dimensions,msoa_household_deprivation_census_2021.csv,Public deprivation proxy for spending-power assumptions.
3,NM_2063_1,TS045 - Car or van availability,msoa_car_van_availability_census_2021.csv,Car access proxy for leisure-trip accessibility.
4,NM_2026_1,TS006 - Population density,msoa_population_density_census_2021.csv,Urban/density context for segmentation.
5,NM_2312_1,SG002 - Approximated Social Grade,msoa_approximated_social_grade_census_2021.csv,Affluence/spending-power proxy for the prototype.


### 2.2 Helper functions for Nomis queries

These helpers build a URL, download a CSV, keep useful columns, and save the raw input. The goal is to make the data acquisition step easy to inspect and easy to change.

In [3]:
def build_nomis_csv_url(dataset_id, category_dimension, category_selector, record_offset=0):
    """Build a Nomis CSV URL for one Census table at 2021 MSOA level.

    Parameters
    ----------
    dataset_id : str
        Nomis dataset ID, for example NM_2020_1 for age bands.
    category_dimension : str
        Table-specific category dimension, for example c2021_age_19.
    category_selector : str
        Nomis category selector. A range such as 0...18 asks for all listed categories.
    record_offset : int
        First row to request. Nomis pages large responses, so we loop over offsets.

    Returns
    -------
    str
        A URL that pandas can read directly as CSV.
    """
    query_params = {
        "date": "latest",
        "geography": MSOA_2021_GEOGRAPHY,
        category_dimension: category_selector,
        "measures": VALUE_MEASURE,
        "recordlimit": NOMIS_PAGE_SIZE,
        "recordoffset": record_offset,
    }

    # urlencode keeps special characters safe inside the query string.
    # We mark the three dots in Nomis ranges as safe so 0...18 remains readable.
    query_string = urlencode(query_params, safe=".")
    return f"{NOMIS_BASE_URL}/{dataset_id}.data.csv?{query_string}"


def read_nomis_table(table_config):
    """Download one configured Nomis table and return it as a DataFrame.

    Nomis includes a RECORD_COUNT column telling us the total rows available for
    the query. We use that to keep requesting pages until the full table has
    been downloaded.
    """
    frames = []
    record_offset = 0
    expected_record_count = None

    while True:
        url = build_nomis_csv_url(
            dataset_id=table_config["dataset_id"],
            category_dimension=table_config["category_dimension"],
            category_selector=table_config["category_selector"],
            record_offset=record_offset,
        )

        # dtype keeps geography/category codes as strings, which prevents leading-zero issues
        # and avoids accidental numeric joins in later sections.
        page_df = pd.read_csv(url, dtype=str)

        # Stop defensively if the API returns an empty page.
        if page_df.empty:
            break

        frames.append(page_df)

        # RECORD_COUNT is repeated on each row; the first row is enough.
        expected_record_count = int(page_df["RECORD_COUNT"].iloc[0])
        record_offset += len(page_df)

        if record_offset >= expected_record_count:
            break

    df = pd.concat(frames, ignore_index=True)

    # OBS_VALUE is the numeric count/value we will model later.
    # Convert it now so quick checks work, while leaving all codes as strings.
    df["OBS_VALUE"] = pd.to_numeric(df["OBS_VALUE"], errors="coerce")

    # Keep the raw Nomis columns for traceability, but add two convenient audit columns.
    df["source_dataset_id"] = table_config["dataset_id"]
    df["source_table_name"] = table_config["table_name"]

    return df


def save_input_table(df, output_file):
    """Save a raw input table to data/input/ as CSV."""
    output_path = DATA_INPUT_DIR / output_file

    # CSV is suitable here because the Nomis tables are rectangular and easy to inspect.
    # We do not save the pandas index because it is not part of the public source data.
    df.to_csv(output_path, index=False)
    return output_path


def create_geography_lookup(age_df):
    """Create a simple MSOA lookup from the age table's total-population rows."""
    total_rows = age_df[age_df["C2021_AGE_19"].eq("0")].copy()

    # The lookup gives later sections one clean place to get the MSOA code/name.
    # Region/local authority can be added later from an ONS lookup if needed.
    lookup = total_rows[["GEOGRAPHY_CODE", "GEOGRAPHY_NAME", "OBS_VALUE"]].rename(
        columns={
            "GEOGRAPHY_CODE": "geo_code",
            "GEOGRAPHY_NAME": "geo_name",
            "OBS_VALUE": "total_population",
        }
    )

    return lookup.sort_values("geo_code").reset_index(drop=True)

### 2.3 Query and save the public MSOA input files

Run this cell to download the configured public tables. If the API is unavailable, the error will happen here and the already-saved CSVs in `data/input/` can still be used.

In [4]:
download_log = []
downloaded_tables = {}

# Check whether all expected raw input files already exist locally.
missing_input_files = [
    table_config["output_file"]
    for table_config in NOMIS_TABLES
    if not (DATA_INPUT_DIR / table_config["output_file"]).exists()
]

if QUERY_NOMIS_DATA:
    print("QUERY_NOMIS_DATA is True: querying Nomis and refreshing data/input/ CSVs.")

    for table_config in NOMIS_TABLES:
        print(f"Downloading {table_config['table_name']}...")

        # Download one long-format Census table at all 2021 MSOAs.
        table_df = read_nomis_table(table_config)

        # Save the raw table immediately so later sections are reproducible.
        output_path = save_input_table(table_df, table_config["output_file"])

        # Keep tables in memory for quick inspection in this notebook session.
        downloaded_tables[table_config["output_file"]] = table_df

        download_log.append(
            {
                "source": "nomis_api",
                "dataset_id": table_config["dataset_id"],
                "table_name": table_config["table_name"],
                "rows": len(table_df),
                "msoa_count": table_df["GEOGRAPHY_CODE"].nunique(),
                "output_path": str(output_path.relative_to(PROJECT_ROOT)),
            }
        )
else:
    print("QUERY_NOMIS_DATA is False: loading existing CSVs from data/input/ without calling Nomis.")

    if missing_input_files:
        missing_files_text = ", ".join(missing_input_files)
        raise FileNotFoundError(
            "QUERY_NOMIS_DATA is False, but these input files are missing from data/input/: "
            f"{missing_files_text}. Set QUERY_NOMIS_DATA = True once to download them."
        )

    for table_config in NOMIS_TABLES:
        output_path = DATA_INPUT_DIR / table_config["output_file"]

        # Load the cached raw CSV exactly as saved from Nomis.
        table_df = pd.read_csv(output_path, dtype=str)
        if "OBS_VALUE" in table_df.columns:
            table_df["OBS_VALUE"] = pd.to_numeric(table_df["OBS_VALUE"], errors="coerce")

        downloaded_tables[table_config["output_file"]] = table_df

        download_log.append(
            {
                "source": "local_csv_cache",
                "dataset_id": table_config["dataset_id"],
                "table_name": table_config["table_name"],
                "rows": len(table_df),
                "msoa_count": table_df["GEOGRAPHY_CODE"].nunique(),
                "output_path": str(output_path.relative_to(PROJECT_ROOT)),
            }
        )

# Create a lightweight geography lookup from the age table, because it has all MSOAs
# and the total population category needed for market-size features later.
age_df = downloaded_tables["msoa_age_5yr_census_2021.csv"]
msoa_lookup = create_geography_lookup(age_df)
lookup_path = save_input_table(msoa_lookup, "msoa_geography_lookup_census_2021.csv")

download_log.append(
    {
        "dataset_id": "derived_from_NM_2020_1",
        "table_name": "MSOA geography lookup with total population",
        "rows": len(msoa_lookup),
        "msoa_count": msoa_lookup["geo_code"].nunique(),
        "output_path": str(lookup_path.relative_to(PROJECT_ROOT)),
    }
)

download_log_df = pd.DataFrame(download_log)
download_log_df

QUERY_NOMIS_DATA is False: loading existing CSVs from data/input/ without calling Nomis.


,source,dataset_id,table_name,rows,msoa_count,output_path
0,local_csv_cache,NM_2020_1,TS007A - Age by five-year age bands,138016,7264,data/input/msoa_age_5yr_census_2021.csv
1,local_csv_cache,NM_2023_1,TS003 - Household composition,108960,7264,data/input/msoa_household_composition_census_2021.csv
2,local_csv_cache,NM_2031_1,TS011 - Households by deprivation dimensions,43584,7264,data/input/msoa_household_deprivation_census_2021.csv
3,local_csv_cache,NM_2063_1,TS045 - Car or van availability,36320,7264,data/input/msoa_car_van_availability_census_2021.csv
4,local_csv_cache,NM_2026_1,TS006 - Population density,7264,7264,data/input/msoa_population_density_census_2021.csv
5,local_csv_cache,NM_2312_1,SG002 - Approximated Social Grade,36320,7264,data/input/msoa_approximated_social_grade_census_2021.csv
6,NaN,derived_from_NM_2020_1,MSOA geography lookup with total population,7264,7264,data/input/msoa_geography_lookup_census_2021.csv


### 2.4 Quick input audit

Before moving on, check that each file downloaded at the same MSOA geography and that the saved CSVs are present. Later sections will do more detailed cleaning and joining.

In [5]:
# List the files created by this section.
saved_input_files = sorted(DATA_INPUT_DIR.glob("*.csv"))
pd.DataFrame(
    {
        "file": [path.name for path in saved_input_files],
        "size_mb": [round(path.stat().st_size / 1_000_000, 2) for path in saved_input_files],
    }
)

,file,size_mb
0,msoa_2021_centroids_ons_geoportal.csv,0.82
1,msoa_age_5yr_census_2021.csv,41.99
2,msoa_approximated_social_grade_census_2021.csv,13.10
3,msoa_car_van_availability_census_2021.csv,11.80
4,msoa_geography_lookup_census_2021.csv,0.23
5,msoa_household_composition_census_2021.csv,39.28
6,msoa_household_deprivation_census_2021.csv,15.09
7,msoa_population_density_census_2021.csv,2.35


In [6]:
# Inspect a few rows from the geography lookup.
# This should show one row per MSOA with geo_code, geo_name, and total_population.
msoa_lookup.head()

,geo_code,geo_name,total_population
0,E02000001,City of London 001,8580
1,E02000002,Barking and Dagenham 001,8286
2,E02000003,Barking and Dagenham 002,11539
3,E02000004,Barking and Dagenham 003,6638
4,E02000005,Barking and Dagenham 004,11082


### 2.5 Add MSOA centroid lookup

We add MSOA centroid coordinates now because they will be needed later for maps, straight-line distance to Merlin attractions, and eventual drive-time/routing work. These coordinates are geographic infrastructure, not customer attributes, so they should **not** be used as clustering variables.

Source: ONS Open Geography Portal, `Middle layer Super Output Areas (December 2021) Boundaries EW BGC (V3)`. The layer includes `LAT`, `LONG`, `BNG_E`, and `BNG_N` attributes for each 2021 MSOA.

- ArcGIS item: https://www.arcgis.com/home/item.html?id=6b282db29762450881ed5159259a6e4e
- Feature service: https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V3/FeatureServer

The notebook caches the lookup as `data/input/msoa_2021_centroids_ons_geoportal.csv`. It refreshes only if the file is missing or `REFRESH_MSOA_CENTROIDS = True`.

In [7]:
MSOA_CENTROID_FILE = "msoa_2021_centroids_ons_geoportal.csv"
MSOA_CENTROID_PATH = DATA_INPUT_DIR / MSOA_CENTROID_FILE

MSOA_CENTROID_QUERY_URL = (
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
    "Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V3/"
    "FeatureServer/0/query"
)


def download_msoa_centroids(page_size=2000):
    """Download 2021 MSOA centroid attributes from the ONS Open Geography Portal.

    We request attributes only and set returnGeometry=false because the later
    distance work needs one representative point per MSOA, not full polygons.
    """
    all_rows = []
    offset = 0

    while True:
        params = {
            "f": "json",
            "where": "1=1",
            "outFields": "MSOA21CD,MSOA21NM,BNG_E,BNG_N,LAT,LONG",
            "returnGeometry": "false",
            "orderByFields": "MSOA21CD",
            "resultOffset": offset,
            "resultRecordCount": page_size,
        }
        response = requests.get(MSOA_CENTROID_QUERY_URL, params=params, timeout=60)
        response.raise_for_status()
        payload = response.json()

        features = payload.get("features", [])
        if not features:
            break

        all_rows.extend(feature["attributes"] for feature in features)

        if len(features) < page_size:
            break
        offset += page_size

    centroid_df = pd.DataFrame(all_rows).rename(
        columns={
            "MSOA21CD": "geo_code",
            "MSOA21NM": "geo_name_centroid_source",
            "BNG_E": "centroid_easting",
            "BNG_N": "centroid_northing",
            "LAT": "latitude",
            "LONG": "longitude",
        }
    )

    centroid_df["centroid_source"] = "ONS Open Geography Portal MSOA 2021 EW BGC V3 LAT/LONG"
    centroid_df = centroid_df[
        [
            "geo_code",
            "geo_name_centroid_source",
            "latitude",
            "longitude",
            "centroid_easting",
            "centroid_northing",
            "centroid_source",
        ]
    ]

    # Convert coordinate columns to numeric so future distance calculations work cleanly.
    for column in ["latitude", "longitude", "centroid_easting", "centroid_northing"]:
        centroid_df[column] = pd.to_numeric(centroid_df[column], errors="coerce")

    return centroid_df


if REFRESH_MSOA_CENTROIDS or not MSOA_CENTROID_PATH.exists():
    print("Refreshing MSOA centroid lookup from the ONS Open Geography Portal.")
    msoa_centroids = download_msoa_centroids()
    msoa_centroids.to_csv(MSOA_CENTROID_PATH, index=False)
else:
    print("Loading cached MSOA centroid lookup from data/input/.")
    msoa_centroids = pd.read_csv(MSOA_CENTROID_PATH, dtype={"geo_code": str})

centroid_audit = {
    "rows": len(msoa_centroids),
    "msoa_count": msoa_centroids["geo_code"].nunique(),
    "missing_latitude": msoa_centroids["latitude"].isna().sum(),
    "missing_longitude": msoa_centroids["longitude"].isna().sum(),
    "output_path": str(MSOA_CENTROID_PATH.relative_to(PROJECT_ROOT)),
}

centroid_audit

Loading cached MSOA centroid lookup from data/input/.


{'rows': 7264,
 'msoa_count': 7264,
 'missing_latitude': 0,
 'missing_longitude': 0,
 'output_path': 'data/input/msoa_2021_centroids_ons_geoportal.csv'}

## 3. Clean and Join Demographic Data

This section converts the raw long-format Nomis downloads into a clean, modelling-ready MSOA table. The output is one row per MSOA with plain-English feature names that can be used for segmentation and opportunity scoring.

The cleaning choices are :

- Keep raw source files unchanged in `data/input/`.
- Create derived features from counts rather than relying on downloaded percentages, so all ratios use the same denominators.
- Join every table on the official MSOA geography code.
- Save the clean joined dataset to `data/processed/msoa_demographic_features.csv` for later sections.

#### Note on household totals across Census tables

Several fields in this section refer to households, but they come from different Census 2021 tables. For example, `total_households` comes from household composition, deprivation fields come from the deprivation table, and car-access fields come from the car or van availability table. Census 2021 uses statistical disclosure control, including small count perturbations, so table totals may differ by a few households even when they describe the same MSOA. For this reason, ratio features such as `deprived_household_share` and `car_access_household_share` are calculated using the denominator from their own source table before we keep the simplified final fields.

### 3.1 Load raw input files

We reload the CSVs from disk so this section can be rerun independently after the data has been downloaded once.

In [8]:
# Store cleaned outputs separately from raw public inputs.
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Centralise file names so later edits only need to happen in one place.
INPUT_FILES = {
    "lookup": "msoa_geography_lookup_census_2021.csv",
    "age": "msoa_age_5yr_census_2021.csv",
    "household": "msoa_household_composition_census_2021.csv",
    "deprivation": "msoa_household_deprivation_census_2021.csv",
    "car": "msoa_car_van_availability_census_2021.csv",
    "density": "msoa_population_density_census_2021.csv",
    "social_grade": "msoa_approximated_social_grade_census_2021.csv",
    "centroids": "msoa_2021_centroids_ons_geoportal.csv",
}


def load_input_csv(file_name):
    """Load one raw input CSV from data/input/.

    We read codes as strings because geography codes are identifiers, not numbers.
    OBS_VALUE is converted later only where we need to calculate with it.
    """
    path = DATA_INPUT_DIR / file_name
    return pd.read_csv(path, dtype=str)


raw_tables = {name: load_input_csv(file_name) for name, file_name in INPUT_FILES.items()}

# Quick audit: each raw table should have the expected 7,264 MSOAs for England and Wales.
raw_input_audit = []
for table_name, df in raw_tables.items():
    geo_column = "geo_code" if "geo_code" in df.columns else "GEOGRAPHY_CODE"
    raw_input_audit.append(
        {
            "table": table_name,
            "rows": len(df),
            "msoa_count": df[geo_column].nunique(),
        }
    )

pd.DataFrame(raw_input_audit)

,table,rows,msoa_count
0,lookup,7264,7264
1,age,138016,7264
2,household,108960,7264
3,deprivation,43584,7264
4,car,36320,7264
5,density,7264,7264
6,social_grade,36320,7264
7,centroids,7264,7264


### 3.2 Helper functions for cleaning Census tables

The Nomis files are long-format tables: each MSOA appears once per category. For modelling, we need wide-format tables where each MSOA is one row and each category is a column.

In [9]:
def pivot_nomis_counts(df, category_column, category_map):
    """Convert a long Nomis count table into one row per MSOA.

    Parameters
    ----------
    df : pandas.DataFrame
        Raw Nomis table with GEOGRAPHY_CODE, GEOGRAPHY_NAME, a category column, and OBS_VALUE.
    category_column : str
        The Nomis category code column to pivot, for example C2021_AGE_19.
    category_map : dict
        Mapping from Nomis category codes to clean column names.

    Returns
    -------
    pandas.DataFrame
        Wide table keyed by geo_code and geo_name.
    """
    keep_columns = ["GEOGRAPHY_CODE", "GEOGRAPHY_NAME", category_column, "OBS_VALUE"]
    working = df[keep_columns].copy()

    # Standardise the geography columns before joining tables together.
    working = working.rename(
        columns={
            "GEOGRAPHY_CODE": "geo_code",
            "GEOGRAPHY_NAME": "geo_name",
            "OBS_VALUE": "value",
        }
    )

    # Keep only the categories we explicitly need. This prevents unused Nomis
    # category codes, such as 1, 11, or 12, from appearing as unclear columns.
    working = working[working[category_column].isin(category_map.keys())].copy()

    # Convert counts/values to numeric. Invalid values become NaN, which makes
    # data-quality issues visible rather than silently turning into strings.
    working["value"] = pd.to_numeric(working["value"], errors="coerce")

    wide = working.pivot_table(
        index=["geo_code", "geo_name"],
        columns=category_column,
        values="value",
        aggfunc="sum",
    ).reset_index()

    # Replace Nomis category codes with readable feature names.
    wide = wide.rename(columns=category_map)

    return wide


def safe_divide(numerator, denominator):
    """Divide two series while avoiding infinite values from zero denominators."""
    result = numerator / denominator.replace({0: pd.NA})
    return result.astype("float64")


def join_feature_table(base_df, feature_df, table_name):
    """Left-join one feature table onto the MSOA base table and report row counts."""
    before_rows = len(base_df)
    joined = base_df.merge(feature_df, on="geo_code", how="left", suffixes=("", f"_{table_name}"))

    # If a joined table also has geo_name, keep the base name and drop the duplicate.
    duplicate_name = f"geo_name_{table_name}"
    if duplicate_name in joined.columns:
        joined = joined.drop(columns=[duplicate_name])

    assert len(joined) == before_rows, f"{table_name} join changed the row count"
    return joined

### 3.3 Create derived demographic features

The derived fields below are designed in a way such that the business logic can be explained to stakeholders. They are public market-potential indicators.

In [10]:
# Start with the geography lookup created in Section 2.
msoa_features = raw_tables["lookup"].copy()
msoa_features["total_population"] = pd.to_numeric(msoa_features["total_population"], errors="coerce")

# Centroids: keep geographic coordinates for mapping and later distance analysis.
# These are useful infrastructure fields, but we should not use latitude/longitude
# as clustering variables because that would cluster places by location rather than demand profile.
centroid_features = raw_tables["centroids"].copy()
for column in ["latitude", "longitude", "centroid_easting", "centroid_northing"]:
    centroid_features[column] = pd.to_numeric(centroid_features[column], errors="coerce")
centroid_features = centroid_features[
    [
        "geo_code",
        "latitude",
        "longitude",
        "centroid_easting",
        "centroid_northing",
        "centroid_source",
    ]
]
msoa_features = join_feature_table(msoa_features, centroid_features, "centroid")

# Age bands: aggregate Census five-year bands into broader groups.
# These groups are simple for stakeholder explanation and future clustering:
# - 0-14: children and family demand potential
# - 15-24: teens/young adults, relevant for thrill-led and city attraction demand
# - 25-44: core parent/decision-maker proxy
# - 45-64: midlife adults, useful market context without over-segmenting
# - 65+: older adults, useful context for areas less centred on family visits
age_map = {
    "0": "total_population_age_table",
    "1": "population_aged_0_4",
    "2": "population_aged_5_9",
    "3": "population_aged_10_14",
    "4": "population_aged_15_19",
    "5": "population_aged_20_24",
    "6": "population_aged_25_29",
    "7": "population_aged_30_34",
    "8": "population_aged_35_39",
    "9": "population_aged_40_44",
    "10": "population_aged_45_49",
    "11": "population_aged_50_54",
    "12": "population_aged_55_59",
    "13": "population_aged_60_64",
    "14": "population_aged_65_69",
    "15": "population_aged_70_74",
    "16": "population_aged_75_79",
    "17": "population_aged_80_84",
    "18": "population_aged_85_plus",
}
age_wide = pivot_nomis_counts(raw_tables["age"], "C2021_AGE_19", age_map)
age_wide["children_count_0_14"] = age_wide[["population_aged_0_4", "population_aged_5_9", "population_aged_10_14"]].sum(axis=1)
age_wide["young_adult_count_15_24"] = age_wide[["population_aged_15_19", "population_aged_20_24"]].sum(axis=1)
age_wide["core_family_adult_count_25_44"] = age_wide[["population_aged_25_29", "population_aged_30_34", "population_aged_35_39", "population_aged_40_44"]].sum(axis=1)
age_wide["midlife_adult_count_45_64"] = age_wide[["population_aged_45_49", "population_aged_50_54", "population_aged_55_59", "population_aged_60_64"]].sum(axis=1)
age_wide["older_adult_count_65_plus"] = age_wide[["population_aged_65_69", "population_aged_70_74", "population_aged_75_79", "population_aged_80_84", "population_aged_85_plus"]].sum(axis=1)
age_wide["children_share_0_14"] = safe_divide(age_wide["children_count_0_14"], age_wide["total_population_age_table"])
age_wide["young_adult_share_15_24"] = safe_divide(age_wide["young_adult_count_15_24"], age_wide["total_population_age_table"])
age_wide["core_family_adult_share_25_44"] = safe_divide(age_wide["core_family_adult_count_25_44"], age_wide["total_population_age_table"])
age_wide["midlife_adult_share_45_64"] = safe_divide(age_wide["midlife_adult_count_45_64"], age_wide["total_population_age_table"])
age_wide["older_adult_share_65_plus"] = safe_divide(age_wide["older_adult_count_65_plus"], age_wide["total_population_age_table"])
age_wide = age_wide[
    [
        "geo_code",
        "geo_name",
        "children_count_0_14",
        "children_share_0_14",
        "young_adult_count_15_24",
        "young_adult_share_15_24",
        "core_family_adult_count_25_44",
        "core_family_adult_share_25_44",
        "midlife_adult_count_45_64",
        "midlife_adult_share_45_64",
        "older_adult_count_65_plus",
        "older_adult_share_65_plus",
    ]
]

# Household composition: keep only total households and households with dependent children.
# We do not split by married/cohabiting/lone-parent status because the prototype
# only needs a simple family-market signal.
household_map = {
    "0": "total_households",
    "5": "households_with_dependent_children_married_couple",
    "8": "households_with_dependent_children_cohabiting_couple",
    "10": "households_with_dependent_children_lone_parent",
    "13": "households_with_dependent_children_other_household",
}
household_wide = pivot_nomis_counts(raw_tables["household"], "C2021_HHCOMP_15", household_map)
dependent_child_household_columns = [
    "households_with_dependent_children_married_couple",
    "households_with_dependent_children_cohabiting_couple",
    "households_with_dependent_children_lone_parent",
    "households_with_dependent_children_other_household",
]
household_wide["households_with_dependent_children"] = household_wide[dependent_child_household_columns].sum(axis=1)
household_wide["family_household_share"] = safe_divide(household_wide["households_with_dependent_children"], household_wide["total_households"])
household_wide = household_wide[
    [
        "geo_code",
        "geo_name",
        "total_households",
        "households_with_dependent_children",
        "family_household_share",
    ]
]

# Deprivation dimensions: aggregate all deprived households into one simple count/share.
# The detailed number of deprivation dimensions is useful for social analysis, but
# too granular for this prototype segmentation.
deprivation_map = {
    "0": "total_households_deprivation_table",
    "1": "households_not_deprived",
    "2": "households_deprived_one_dimension",
    "3": "households_deprived_two_dimensions",
    "4": "households_deprived_three_dimensions",
    "5": "households_deprived_four_dimensions",
}
deprivation_wide = pivot_nomis_counts(raw_tables["deprivation"], "C2021_DEP_6", deprivation_map)
deprivation_columns = [
    "households_deprived_one_dimension",
    "households_deprived_two_dimensions",
    "households_deprived_three_dimensions",
    "households_deprived_four_dimensions",
]
deprivation_wide["households_deprived"] = deprivation_wide[deprivation_columns].sum(axis=1)
deprivation_wide["not_deprived_household_share"] = safe_divide(deprivation_wide["households_not_deprived"], deprivation_wide["total_households_deprivation_table"])
deprivation_wide["deprived_household_share"] = safe_divide(deprivation_wide["households_deprived"], deprivation_wide["total_households_deprivation_table"])
deprivation_wide = deprivation_wide[
    [
        "geo_code",
        "geo_name",
        "households_deprived",
        "deprived_household_share",
        "households_not_deprived",
        "not_deprived_household_share",
    ]
]

# Car availability: aggregate to whether a household has at least one car or van.
# We do not need the number of cars for this prototype; a simple access flag is
# enough for broad leisure-trip practicality.
car_map = {
    "0": "total_households_car_table",
    "1": "households_no_car_or_van",
    "2": "households_one_car_or_van",
    "3": "households_two_cars_or_vans",
    "4": "households_three_plus_cars_or_vans",
}
car_wide = pivot_nomis_counts(raw_tables["car"], "C2021_CARS_5", car_map)
car_wide["households_with_car_or_van"] = car_wide[["households_one_car_or_van", "households_two_cars_or_vans", "households_three_plus_cars_or_vans"]].sum(axis=1)
car_wide["car_access_household_share"] = safe_divide(car_wide["households_with_car_or_van"], car_wide["total_households_car_table"])
car_wide = car_wide[
    [
        "geo_code",
        "geo_name",
        "households_no_car_or_van",
        "households_with_car_or_van",
        "car_access_household_share",
    ]
]

# Population density is already one value per MSOA.
density_wide = pivot_nomis_counts(raw_tables["density"], "CELL", {"0": "population_density_per_sq_km"})

# Social grade: AB/C1 is a simple public proxy for higher spending power.
social_grade_map = {
    "0": "total_residents_social_grade_table",
    "1": "residents_social_grade_ab",
    "2": "residents_social_grade_c1",
    "3": "residents_social_grade_c2",
    "4": "residents_social_grade_de",
}
social_grade_wide = pivot_nomis_counts(raw_tables["social_grade"], "C2021_ASG_5", social_grade_map)
social_grade_wide["residents_social_grade_ab_c1"] = social_grade_wide[["residents_social_grade_ab", "residents_social_grade_c1"]].sum(axis=1)
social_grade_wide["ab_c1_social_grade_share"] = safe_divide(social_grade_wide["residents_social_grade_ab_c1"], social_grade_wide["total_residents_social_grade_table"])
social_grade_wide = social_grade_wide[
    [
        "geo_code",
        "geo_name",
        "residents_social_grade_ab_c1",
        "ab_c1_social_grade_share",
    ]
]

# Join the feature groups into a single MSOA table.
for table_name, feature_df in {
    "age": age_wide,
    "household": household_wide,
    "deprivation": deprivation_wide,
    "car": car_wide,
    "density": density_wide,
    "social_grade": social_grade_wide,
}.items():
    msoa_features = join_feature_table(msoa_features, feature_df, table_name)

print("Dateset preview:")
display(msoa_features.drop(columns=[col for col in msoa_features.columns if "share" in col]).head().T)

print("\nDataset preview (with share variables):")
display(msoa_features.head().T)

Dateset preview:


,0,1,2,3,4
geo_code,E02000001,E02000002,E02000003,E02000004,E02000005
geo_name,City of London 001,Barking and Dagenham 001,Barking and Dagenham 002,Barking and Dagenham 003,Barking and Dagenham 004
total_population,8580,8286,11539,6638,11082
latitude,51.51562,51.58652,51.57606,51.55639,51.56069
longitude,-0.09349,0.138756,0.138149,0.176828,0.144267
centroid_easting,532384,548267,548259,551004,548733
centroid_northing,181355,189685,188520,186412,186824
centroid_source,ONS Open Geography Portal MSOA 2021 EW BGC V3 LAT/LONG,ONS Open Geography Portal MSOA 2021 EW BGC V3 LAT/LONG,ONS Open Geography Portal MSOA 2021 EW BGC V3 LAT/LONG,ONS Open Geography Portal MSOA 2021 EW BGC V3 LAT/LONG,ONS Open Geography Portal MSOA 2021 EW BGC V3 LAT/LONG
children_count_0_14,546,2172,2560,1238,2935
young_adult_count_15_24,1181,1028,1451,815,1355



Dataset preview (with share variables):


,0,1,2,3,4
geo_code,E02000001,E02000002,E02000003,E02000004,E02000005
geo_name,City of London 001,Barking and Dagenham 001,Barking and Dagenham 002,Barking and Dagenham 003,Barking and Dagenham 004
total_population,8580,8286,11539,6638,11082
latitude,51.51562,51.58652,51.57606,51.55639,51.56069
longitude,-0.09349,0.138756,0.138149,0.176828,0.144267
centroid_easting,532384,548267,548259,551004,548733
centroid_northing,181355,189685,188520,186412,186824
centroid_source,ONS Open Geography Portal MSOA 2021 EW BGC V3 LAT/LONG,ONS Open Geography Portal MSOA 2021 EW BGC V3 LAT/LONG,ONS Open Geography Portal MSOA 2021 EW BGC V3 LAT/LONG,ONS Open Geography Portal MSOA 2021 EW BGC V3 LAT/LONG,ONS Open Geography Portal MSOA 2021 EW BGC V3 LAT/LONG
children_count_0_14,546,2172,2560,1238,2935
children_share_0_14,0.063636,0.262129,0.221856,0.186502,0.264844


### 3.4 Data quality checks and export

Before using this table for clustering, check row counts, missing values, and a few derived feature ranges. This gives us an audit trail for stakeholder explanation.

In [11]:
# Select the main columns that will be most useful for the next modelling steps.
model_feature_columns = [
    "geo_code",
    "geo_name",
    "latitude",
    "longitude",
    "centroid_easting",
    "centroid_northing",
    "centroid_source",
    "total_population",
    "children_count_0_14",
    "children_share_0_14",
    "young_adult_count_15_24",
    "young_adult_share_15_24",
    "core_family_adult_count_25_44",
    "core_family_adult_share_25_44",
    "midlife_adult_count_45_64",
    "midlife_adult_share_45_64",
    "older_adult_count_65_plus",
    "older_adult_share_65_plus",
    "total_households",
    "households_with_dependent_children",
    "family_household_share",
    "households_deprived",
    "deprived_household_share",
    "households_not_deprived",
    "not_deprived_household_share",
    "households_no_car_or_van",
    "households_with_car_or_van",
    "car_access_household_share",
    "population_density_per_sq_km",
    "residents_social_grade_ab_c1",
    "ab_c1_social_grade_share",
]

msoa_demographic_features = msoa_features[model_feature_columns].copy()

# Add a simple region placeholder from the first letter of the MSOA code.
# E-codes are England and W-codes are Wales in these Census 2021 outputs.
msoa_demographic_features["country"] = msoa_demographic_features["geo_code"].str[0].map({"E": "England", "W": "Wales"})

# Summarise missingness so we can spot join or source-data problems quickly.
missingness_summary = (
    msoa_demographic_features.isna()
    .sum()
    .reset_index(name="missing_rows")
    .rename(columns={"index": "column"})
)
missingness_summary["missing_share"] = missingness_summary["missing_rows"] / len(msoa_demographic_features)

# Summarise core numeric fields for a quick reasonableness check.
numeric_summary = msoa_demographic_features.drop(columns=["geo_code", "geo_name", "country"]).describe().T

# Save the cleaned joined dataset for the next notebook section.
clean_output_path = DATA_PROCESSED_DIR / "msoa_demographic_features.csv"
msoa_demographic_features.to_csv(clean_output_path, index=False)

print(f"Saved clean demographic features to: {clean_output_path.relative_to(PROJECT_ROOT)}")
print(f"Rows: {len(msoa_demographic_features):,}")
print(f"Columns: {len(msoa_demographic_features.columns):,}")

missingness_summary

Saved clean demographic features to: data/processed/msoa_demographic_features.csv
Rows: 7,264
Columns: 32


,column,missing_rows,missing_share
0,geo_code,0,0.0
1,geo_name,0,0.0
2,latitude,0,0.0
3,longitude,0,0.0
4,centroid_easting,0,0.0
5,centroid_northing,0,0.0
6,centroid_source,0,0.0
7,total_population,0,0.0
8,children_count_0_14,0,0.0
9,children_share_0_14,0,0.0


In [12]:
# Inspect the numeric feature ranges. Shares should generally sit between 0 and 1.
numeric_summary

,count,mean,std,min,25%,50%,75%,max
latitude,7264.0,52.351519,1.127218,49.923330,51.476983,52.119545,53.367415,55.759030
longitude,7264.0,-1.346497,1.302268,-6.302170,-2.215653,-1.374700,-0.244310,1.744019
centroid_easting,7264.0,445038.238161,89867.042194,91327.000000,385582.000000,441983.000000,521361.750000,654401.000000
centroid_northing,7264.0,273620.237197,124895.014224,11447.000000,176624.750000,248519.500000,385920.500000,651742.000000
total_population,7264.0,8204.482792,1815.668163,2055.000000,6827.000000,7955.500000,9289.250000,18478.000000
children_count_0_14,7264.0,1425.195760,479.862310,136.000000,1086.000000,1353.000000,1685.250000,4313.000000
children_share_0_14,7264.0,0.172368,0.035719,0.014016,0.150939,0.170559,0.192168,0.368466
young_adult_count_15_24,7264.0,963.210077,647.829161,124.000000,676.000000,829.000000,1041.000000,12504.000000
young_adult_share_15_24,7264.0,0.114822,0.055603,0.048405,0.092798,0.102599,0.116907,0.782135
core_family_adult_count_25_44,7264.0,2173.428001,812.353523,487.000000,1584.000000,2021.000000,2605.250000,6738.000000


### 3.5 Data dictionary for the cleaned MSOA dataset

This data dictionary briefly explains each column in `msoa_demographic_features.csv`. It is saved alongside the processed dataset so later notebooks and stakeholder documentation can reuse the same definitions.

In [13]:
# Build a compact data dictionary for the final cleaned dataset.
# Each row describes one output column, using plain language that can be shared
# with stakeholders without requiring them to read the cleaning code above.
data_dictionary_rows = [
    {"column": "geo_code", "description": "MSOA 2021 geography code. Each code represents one Middle layer Super Output Area in England or Wales.", "source_or_derivation": "Census 2021 MSOA geography from Nomis"},
    {"column": "geo_name", "description": "MSOA 2021 geography name.", "source_or_derivation": "Census 2021 MSOA geography from Nomis"},
    {"column": "latitude", "description": "Latitude of the MSOA centroid in WGS84 coordinates.", "source_or_derivation": "ONS Open Geography Portal MSOA 2021 centroid attributes"},
    {"column": "longitude", "description": "Longitude of the MSOA centroid in WGS84 coordinates.", "source_or_derivation": "ONS Open Geography Portal MSOA 2021 centroid attributes"},
    {"column": "centroid_easting", "description": "British National Grid easting for the MSOA centroid.", "source_or_derivation": "ONS Open Geography Portal MSOA 2021 centroid attributes"},
    {"column": "centroid_northing", "description": "British National Grid northing for the MSOA centroid.", "source_or_derivation": "ONS Open Geography Portal MSOA 2021 centroid attributes"},
    {"column": "centroid_source", "description": "Text label recording the source used for the centroid coordinates.", "source_or_derivation": "Added during centroid lookup step"},
    {"column": "total_population", "description": "Total usual resident population in the MSOA.", "source_or_derivation": "Census 2021 population lookup from Nomis"},
    {"column": "children_count_0_14", "description": "Number of residents aged 0 to 14.", "source_or_derivation": "Sum of Census 2021 five-year age bands 0-4, 5-9, and 10-14"},
    {"column": "children_share_0_14", "description": "Share of residents aged 0 to 14.", "source_or_derivation": "children_count_0_14 divided by total population in the age table"},
    {"column": "young_adult_count_15_24", "description": "Number of residents aged 15 to 24.", "source_or_derivation": "Sum of Census 2021 five-year age bands 15-19 and 20-24"},
    {"column": "young_adult_share_15_24", "description": "Share of residents aged 15 to 24.", "source_or_derivation": "young_adult_count_15_24 divided by total population in the age table"},
    {"column": "core_family_adult_count_25_44", "description": "Number of residents aged 25 to 44, used as a broad proxy for core parent and decision-maker age groups.", "source_or_derivation": "Sum of Census 2021 five-year age bands 25-29, 30-34, 35-39, and 40-44"},
    {"column": "core_family_adult_share_25_44", "description": "Share of residents aged 25 to 44.", "source_or_derivation": "core_family_adult_count_25_44 divided by total population in the age table"},
    {"column": "midlife_adult_count_45_64", "description": "Number of residents aged 45 to 64.", "source_or_derivation": "Sum of Census 2021 five-year age bands 45-49, 50-54, 55-59, and 60-64"},
    {"column": "midlife_adult_share_45_64", "description": "Share of residents aged 45 to 64.", "source_or_derivation": "midlife_adult_count_45_64 divided by total population in the age table"},
    {"column": "older_adult_count_65_plus", "description": "Number of residents aged 65 and over.", "source_or_derivation": "Sum of Census 2021 five-year age bands from 65-69 through 90+"},
    {"column": "older_adult_share_65_plus", "description": "Share of residents aged 65 and over.", "source_or_derivation": "older_adult_count_65_plus divided by total population in the age table"},
    {"column": "total_households", "description": "Total number of households in the MSOA.", "source_or_derivation": "Census 2021 household composition table from Nomis"},
    {"column": "households_with_dependent_children", "description": "Number of households with dependent children, regardless of parental marital status.", "source_or_derivation": "Sum of relevant Census 2021 household composition categories"},
    {"column": "family_household_share", "description": "Share of households that have dependent children.", "source_or_derivation": "households_with_dependent_children divided by total_households"},
    {"column": "households_deprived", "description": "Number of households deprived in at least one measured dimension.", "source_or_derivation": "Sum of Census 2021 deprivation categories for one or more deprivation dimensions"},
    {"column": "deprived_household_share", "description": "Share of households deprived in at least one measured dimension.", "source_or_derivation": "households_deprived divided by the deprivation table household total"},
    {"column": "households_not_deprived", "description": "Number of households not deprived in any measured dimension.", "source_or_derivation": "Census 2021 household deprivation category: not deprived in any dimension"},
    {"column": "not_deprived_household_share", "description": "Share of households not deprived in any measured dimension.", "source_or_derivation": "households_not_deprived divided by the deprivation table household total"},
    {"column": "households_no_car_or_van", "description": "Number of households with no car or van available.", "source_or_derivation": "Census 2021 car or van availability table from Nomis"},
    {"column": "households_with_car_or_van", "description": "Number of households with at least one car or van available.", "source_or_derivation": "Sum of Census 2021 car availability categories for one, two, or three-plus cars/vans"},
    {"column": "car_access_household_share", "description": "Share of households with at least one car or van available.", "source_or_derivation": "households_with_car_or_van divided by the car availability table household total"},
    {"column": "population_density_per_sq_km", "description": "Population density measured as residents per square kilometre.", "source_or_derivation": "Census 2021 population density table from Nomis"},
    {"column": "residents_social_grade_ab_c1", "description": "Number of residents in approximated social grades AB or C1.", "source_or_derivation": "Sum of Census 2021 approximated social grade AB and C1 categories"},
    {"column": "ab_c1_social_grade_share", "description": "Share of residents in approximated social grades AB or C1, used as a simple public proxy for higher spending power.", "source_or_derivation": "residents_social_grade_ab_c1 divided by the social grade table total"},
    {"column": "country", "description": "Country label for the MSOA.", "source_or_derivation": "Derived from the first letter of geo_code: E = England, W = Wales"},
]

data_dictionary = pd.DataFrame(data_dictionary_rows)

# Confirm the dictionary covers every column in the processed dataset exactly once.
missing_dictionary_columns = sorted(set(msoa_demographic_features.columns) - set(data_dictionary["column"]))
extra_dictionary_columns = sorted(set(data_dictionary["column"]) - set(msoa_demographic_features.columns))
assert not missing_dictionary_columns, f"Columns missing from data dictionary: {missing_dictionary_columns}"
assert not extra_dictionary_columns, f"Dictionary columns not found in dataset: {extra_dictionary_columns}"

# Save the dictionary beside the cleaned dataset so both artefacts travel together.
data_dictionary_path = DATA_PROCESSED_DIR / "msoa_demographic_features_data_dictionary.csv"
data_dictionary.to_csv(data_dictionary_path, index=False)

print(f"Saved data dictionary to: {data_dictionary_path.relative_to(PROJECT_ROOT)}")
data_dictionary

Saved data dictionary to: data/processed/msoa_demographic_features_data_dictionary.csv


,column,description,source_or_derivation
0,geo_code,MSOA 2021 geography code. Each code represents one Middle layer Super Output Area in England or Wales.,Census 2021 MSOA geography from Nomis
1,geo_name,MSOA 2021 geography name.,Census 2021 MSOA geography from Nomis
2,latitude,Latitude of the MSOA centroid in WGS84 coordinates.,ONS Open Geography Portal MSOA 2021 centroid attributes
3,longitude,Longitude of the MSOA centroid in WGS84 coordinates.,ONS Open Geography Portal MSOA 2021 centroid attributes
4,centroid_easting,British National Grid easting for the MSOA centroid.,ONS Open Geography Portal MSOA 2021 centroid attributes
5,centroid_northing,British National Grid northing for the MSOA centroid.,ONS Open Geography Portal MSOA 2021 centroid attributes
6,centroid_source,Text label recording the source used for the centroid coordinates.,Added during centroid lookup step
7,total_population,Total usual resident population in the MSOA.,Census 2021 population lookup from Nomis
8,children_count_0_14,Number of residents aged 0 to 14.,"Sum of Census 2021 five-year age bands 0-4, 5-9, and 10-14"
9,children_share_0_14,Share of residents aged 0 to 14.,children_count_0_14 divided by total population in the age table


## 4. Import Merlin attraction data

This section imports the Merlin attraction dataset that we will later use for geographic access, attraction fit, and opportunity scoring.

In [14]:
# Load the curated Merlin attraction dataset from the processed data folder.
# This file is separate from the public Census inputs because it describes Merlin
# locations and attraction characteristics rather than MSOA demographics.
MERLIN_ATTRACTION_PATH = DATA_PROCESSED_DIR / "merlin_attraction_data.csv"

merlin_attractions = pd.read_csv(MERLIN_ATTRACTION_PATH)

# Basic audit so we can quickly confirm that the file loaded as expected.
merlin_attraction_audit = {
    "rows": len(merlin_attractions),
    "columns": len(merlin_attractions.columns),
    "source_path": str(MERLIN_ATTRACTION_PATH.relative_to(PROJECT_ROOT)),
}

merlin_attraction_audit

{'rows': 20,
 'columns': 11,
 'source_path': 'data/processed/merlin_attraction_data.csv'}

### 4.1 Preview all Merlin attractions

The attraction table is small, so we can display every row in the notebook.

In [15]:
# Show every attraction row. The temporary display option prevents pandas from
# truncating the table if the attraction list grows slightly during prototyping.
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(merlin_attractions)

,attraction_name_official,brand_official,business_division_official,city_official,postcode_official,latitude,longitude,experience_category_inferred,indoor_outdoor_inferred,target_age_group_inferred,family_friendly_inferred
0,Alton Towers Resort,Alton Towers,Resort Theme Parks,Alton,ST10 4DB,52.985926,-1.887592,Resort Theme Park,Mixed,children_0_14|young_adults_15_24|core_family_adults_25_44,Yes
1,Thorpe Park,Thorpe Park,Resort Theme Parks,Chertsey,KT16 8PN,51.405100,-0.532991,Resort Theme Park,Outdoor,young_adults_15_24|core_family_adults_25_44,Yes
2,Chessington World of Adventures Resort,Chessington,Resort Theme Parks,Chessington,KT9 2NE,51.348502,-0.321584,Resort Theme Park,Mixed,children_0_14|core_family_adults_25_44,Yes
3,LEGOLAND Windsor Resort,LEGOLAND,Resort Theme Parks,Windsor,SL4 4AY,51.463834,-0.652602,Resort Theme Park,Outdoor,children_0_14|core_family_adults_25_44,Yes
4,Warwick Castle,Warwick Castle,Gateway Attractions,Warwick,CV34 6AU,52.278247,1.591054,Heritage & Castle,Mixed,children_0_14|core_family_adults_25_44|midlife_adults_45_64,Yes
5,London Eye,London Eye,Gateway Attractions,London,SE1 7PB,51.503186,-0.122094,Observation Attraction,Outdoor,all_age_groups,Yes
6,Madame Tussauds London,Madame Tussauds,Gateway Attractions,London,NW1 5LR,51.523005,-0.156988,Celebrity & Museum,Indoor,all_age_groups,Yes
7,London Dungeon,The Dungeons,Gateway Attractions,London,SE1 7PB,51.502512,-0.121338,Immersive Entertainment,Indoor,young_adults_15_24|core_family_adults_25_44,No
8,Shrek's Adventure! London,Shrek's Adventure!,Gateway Attractions,London,SE1 7PB,51.501882,-0.121948,Immersive Entertainment,Indoor,children_0_14|core_family_adults_25_44,Yes
9,Cadbury World,Cadbury World,Gateway Attractions,Birmingham,B30 1JR,52.428795,-1.934000,Brand Experience,Indoor,children_0_14|core_family_adults_25_44|midlife_adults_45_64,Yes


## 5. Import competitor attraction data

This section imports the curated competitor attraction dataset. We keep this step simple for now: load the file, run a small audit, and preview the full table.

In [16]:
# Load the curated competitor attraction dataset.
# This file describes major UK competitor attractions and their broad audience fit.
COMPETITOR_ATTRACTION_PATH = DATA_PROCESSED_DIR / "merlin_competitor_data.csv"

competitor_attractions = pd.read_csv(COMPETITOR_ATTRACTION_PATH)

competitor_attraction_audit = {
    "rows": len(competitor_attractions),
    "columns": len(competitor_attractions.columns),
    "source_path": str(COMPETITOR_ATTRACTION_PATH.relative_to(PROJECT_ROOT)),
}

competitor_attraction_audit

{'rows': 32,
 'columns': 13,
 'source_path': 'data/processed/merlin_competitor_data.csv'}

### 5.1 Preview all competitor attractions

The competitor table is small, so we display every row in the notebook.

In [17]:
# Show every competitor attraction row. The temporary display option prevents
# pandas from truncating columns or rows during inspection.
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(competitor_attractions)

,attraction_name,operator,attraction_category,attraction_type,city,postcode,family_friendly,indoor_outdoor,children_count_0_14,young_adult_count_15_24,core_family_adult_count_25_44,midlife_adult_count_45_64,older_adult_count_65_plus
0,Paultons Park,Paultons Park Ltd,Theme Park,Theme Park,Romsey,SO51 6AL,Yes,Outdoor,1,0,1,1,0
1,Drayton Manor Resort,Looping Group,Theme Park Resort,Theme Park,Tamworth,B78 3TW,Yes,Outdoor,1,0,1,1,0
2,Flamingo Land Resort,Flamingo Land,Theme Park & Zoo,Theme Park + Zoo,Malton,YO17 6UX,Yes,Mixed,1,0,1,1,0
3,Blackpool Pleasure Beach,Pleasure Beach Resort,Theme Park,Theme Park,Blackpool,FY4 1EZ,Partial,Outdoor,0,1,1,1,0
4,Adventure Island,Stockvale Group,Theme Park,Theme Park,Southend-on-Sea,SS1 1EE,Yes,Outdoor,1,0,1,1,0
5,Gulliver's World,Gulliver's,Theme Park,Family Theme Park,Warrington,WA5 9YZ,Yes,Outdoor,1,0,1,0,0
6,Gulliver's Kingdom,Gulliver's,Theme Park,Family Theme Park,Matlock Bath,DE4 3PG,Yes,Outdoor,1,0,1,0,0
7,Gulliver's Land,Gulliver's,Theme Park,Family Theme Park,Milton Keynes,MK15 0DT,Yes,Outdoor,1,0,1,0,0
8,Dreamland Margate,Dreamland Trust,Amusement Park,Amusement Park,Margate,CT9 1XJ,Yes,Outdoor,0,1,1,1,0
9,Chester Zoo,North of England Zoological Society,Zoo,Zoo,Chester,CH2 1EU,Yes,Mixed,1,1,1,1,1
